<a href="https://colab.research.google.com/github/RB1Student/ISYS2001-Assignment-2-Project/blob/main/SmartStatement%20%20Bank%20Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
# =====================================================
#  BANK STATEMENT APP: ACCOUNT + DASHBOARD + CHATBOT
#  Paste this whole thing into ONE Colab cell and run it.
#  Your CSV needs these columns: Date, Name, Income, Expense, Balance
# =====================================================
import io
import html
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import userdata
from google import genai
from google.genai import types

# ---------- 0. Connect to Gemini ----------
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))   # key from Colab's 🔑 Secrets
MODEL = "gemini-3.5-flash"   # check AI Studio for the current model name


# ---------- 1. Loading + cleaning the CSV ----------
def read_uploaded_csv(upload_widget):
    """Get the uploaded file and turn it into a table (DataFrame)."""
    value = upload_widget.value

    # Older ipywidgets give a dict {filename: file}, newer ones give a tuple of files
    if isinstance(value, dict):
        filename = list(value.keys())[0]
        file_info = value[filename]
    else:
        file_info = value[0]
        filename = file_info["name"]

    # The file arrives as bytes, so wrap it in BytesIO so pandas can read it like a file
    df = pd.read_csv(io.BytesIO(bytes(file_info["content"])))
    return df, filename


def clean_money(series):
    """Turn text like '$1,234.50' or '—' into numbers like 1234.5 or 0."""
    text = series.astype(str).str.replace(r"[^\d.\-]", "", regex=True)   # keep only digits . and -
    return pd.to_numeric(text, errors="coerce").fillna(0)                # blanks become 0


def clean_statement(df):
    """Clean the money columns, fix the dates and add a Month column."""
    for col in ["Income", "Expense", "Balance"]:
        df[col] = clean_money(df[col])

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)   # 05/10/2026 = 5 October
    df["Month"] = df["Date"].dt.strftime("%B %Y")             # e.g. "October 2026"
    return df


# ---------- 2. Dashboard pieces ----------
def kpi_card(title, value, colour="#1a237e"):
    """One coloured box showing a title and a big number."""
    return f"""
    <div style="display:inline-block; margin:6px; padding:12px 18px;
                border:2px solid {colour}; border-radius:10px; min-width:160px;">
        <div style="color:#555; font-size:13px;">{title}</div>
        <div style="color:{colour}; font-size:22px; font-weight:bold;">{value}</div>
    </div>
    """


def plot_charts(df, top_n=8):
    """Bar chart (monthly expense) and pie chart (expense by category) side by side."""
    fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(16, 6))

    # --- Bar chart: total expense per month ---
    monthly_expense = df.groupby("Month", sort=False)["Expense"].sum()
    monthly_expense.plot(kind="bar", color="skyblue", ax=ax_bar)
    ax_bar.set_title("Monthly Expenses")
    ax_bar.set_ylabel("Total Expense ($)")
    ax_bar.tick_params(axis="x", rotation=45)

    # --- Pie chart: total expense per category, biggest first ---
    category_expense = df.groupby("Name")["Expense"].sum().sort_values(ascending=False)
    category_expense = category_expense[category_expense > 0]

    if category_expense.empty:
        ax_pie.text(0.5, 0.5, "No expenses to show", ha="center", va="center")
        ax_pie.axis("off")
    else:
        # Keep the biggest categories and add the rest together as "Other"
        if len(category_expense) > top_n:
            other = category_expense.iloc[top_n:].sum()
            category_expense = category_expense.iloc[:top_n]
            category_expense["Other"] = other

        wedges, _, _ = ax_pie.pie(category_expense, autopct="%1.1f%%", startangle=90,
                                  counterclock=False, wedgeprops={"edgecolor": "white"})

        # Legend under the pie, e.g. "Groceries - $412.30"
        labels = [f"{name} - ${amount:,.2f}" for name, amount in category_expense.items()]
        ax_pie.legend(wedges, labels, title=f"Total: ${category_expense.sum():,.2f}",
                      loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
        ax_pie.set_title("Expenses by Category")

    plt.tight_layout()
    plt.show()


def display_financial_dashboard(df, user):
    """Show the KPI cards, the charts and the transaction table."""
    balance = df["Balance"].iloc[-1]      # balance on the last row
    income = df["Income"].sum()
    expense = df["Expense"].sum()
    net = income - expense
    net_colour = "#2e7d32" if net >= 0 else "#c62828"   # green if positive, red if negative

    display(HTML(f"""
        <h2>{html.escape(user)}'s Financial Dashboard</h2>
        <p>Showing {len(df) - 1} transaction(s)</p>
        {kpi_card("Current Balance", f"${balance:,.2f}", "#1a237e")}
        {kpi_card("Total Income", f"${income:,.2f}", "#2e7d32")}
        {kpi_card("Total Expense", f"${expense:,.2f}", "#c62828")}
        {kpi_card("Net (Income - Expense)", f"${net:,.2f}", net_colour)}
    """))

    plot_charts(df)

    display(HTML("<h3>Transactions</h3>"))
    display(df.drop(columns="Month").reset_index(drop=True))


def build_dashboard_screen(df, user):
    """The Dashboard tab: two dropdown filters with the dashboard underneath."""
    month_dd = widgets.Dropdown(options=["All"] + list(df["Month"].unique()), description="Month:")
    category_dd = widgets.Dropdown(options=["All"] + sorted(df["Name"].unique()), description="Category:")
    output = widgets.Output()

    def update(change=None):
        # Keep only the rows that match the chosen filters
        filtered = df
        if month_dd.value != "All":
            filtered = filtered[filtered["Month"] == month_dd.value]
        if category_dd.value != "All":
            filtered = filtered[filtered["Name"] == category_dd.value]

        # Redraw the dashboard inside the output area
        with output:
            output.clear_output(wait=True)
            if filtered.empty:
                display(HTML("<p>No transactions match those filters. Please adjust your selections.</p>"))
            else:
                display_financial_dashboard(filtered, user)

    month_dd.observe(update, names="value")      # redraw when the month changes
    category_dd.observe(update, names="value")   # redraw when the category changes
    update()                                     # show the dashboard straight away

    return widgets.VBox([widgets.HBox([month_dd, category_dd]), output])


# ---------- 3. Chatbot screen ----------
def build_chat_screen(df, user):
    """The Chatbot tab: Gemini answers questions about the statement."""
    statement_text = df.drop(columns="Month").to_csv(index=False)
    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=(
                f"You are a friendly finance assistant talking to {user}. "
                "Answer using only this bank statement:\n\n" + statement_text
            )
        ),
    )

    chat_area = widgets.Output()
    text_box = widgets.Text(placeholder="Ask about your spending...",
                            layout=widgets.Layout(width="500px"))
    send_btn = widgets.Button(description="Send", button_style="primary")

    def send(_):
        question = text_box.value.strip()
        if not question:
            return
        text_box.value = ""
        with chat_area:
            print(f"You: {question}")
            try:
                reply = chat.send_message(question).text
            except Exception as error:
                reply = f"Something went wrong talking to the API: {error}"
            print(f"Gemini: {reply}\n")

    send_btn.on_click(send)
    text_box.on_submit(send)   # press Enter to send

    return widgets.VBox([
        widgets.HTML(f"<h3>Hi {html.escape(user)}! Ask me about your bank statement</h3>"),
        chat_area,
        widgets.HBox([text_box, send_btn]),
    ])


# ---------- 4. Account screen (log in / log off) ----------
def message_box(text, colour="#e65100", background="#fff3e0"):
    """A coloured message box (orange warning by default)."""
    return (f'<div style="color:{colour}; background:{background}; '
            f'padding:10px; border-radius:6px;">{text}</div>')


def placeholder(text):
    """A blue message box widget for tabs that are waiting for log in."""
    return widgets.HTML(message_box(text, colour="#1a237e", background="#e8eaf6"))


# One empty container per tab. We swap what's inside them as the user logs in and out.
account_box = widgets.VBox()
dashboard_box = widgets.VBox()
chat_box = widgets.VBox()


def show_login_screen():
    """Show the log in form and put 'please log in' messages on the other tabs."""
    name_box = widgets.Text(placeholder="Your name", description="Name:")
    upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
    start_btn = widgets.Button(description="Load my statement", button_style="success", icon="check")
    clear_btn = widgets.Button(description="Clear", button_style="warning", icon="times")   # NEW
    message = widgets.HTML()

    def start(_):
        name = name_box.value.strip()
        if not name:
            message.value = message_box("Please enter your name.")
            return
        if not upload.value:
            message.value = message_box("Please upload your bank statement CSV.")
            return
        try:
            df, filename = read_uploaded_csv(upload)
            df = clean_statement(df)
        except Exception as error:
            message.value = message_box(f"Couldn't read that file: {html.escape(str(error))}")
            return
        log_in(name, df, filename)

    def clear(_):                                   # NEW
        """Reset the form so a different CSV can be uploaded."""
        show_login_screen()

    start_btn.on_click(start)
    clear_btn.on_click(clear)                       # NEW

    account_box.children = [
        widgets.HTML("<h3>Welcome! Let's get started</h3>"),
        name_box,
        widgets.HTML("<h4>Format Columns headings to | Date | Name | Type | Income | Expense | Balance |</h4>"),
        upload,
        widgets.HBox([start_btn, clear_btn]),       # NEW (replaces the old start_btn line)
        message,
    ]
    waiting = "Please enter your name and upload your CSV on the 👤 Account tab first."
    dashboard_box.children = [placeholder(waiting)]
    chat_box.children = [placeholder(waiting)]


def log_in(name, df, filename):
    """Save the user's data and build the Dashboard and Chatbot tabs."""
    global User, Bank_Statement     # so you can use these in other cells too
    User = name
    Bank_Statement = df

    logoff_btn = widgets.Button(description="Log off", button_style="danger", icon="sign-out")
    logoff_btn.on_click(log_off)

    account_box.children = [
        widgets.HTML(message_box(
            f"Logged in as <b>{html.escape(name)}</b><br>"
            f"File: {html.escape(filename)} ({len(df)} transactions)",
            colour="#2e7d32", background="#e8f5e9",
        )),
        logoff_btn,
    ]
    dashboard_box.children = [build_dashboard_screen(df, name)]
    chat_box.children = [build_chat_screen(df, name)]
    tabs.selected_index = 1   # jump to the Dashboard tab


def log_off(_):
    """Forget the user's data and go back to the log in form."""
    global User, Bank_Statement
    User = None
    Bank_Statement = None
    show_login_screen()       # resets all three tabs
    tabs.selected_index = 0   # back to the Account tab


# ---------- 5. Tabs ----------
tabs = widgets.Tab(children=[account_box, dashboard_box, chat_box])
tabs.set_title(0, "👤 Account")
tabs.set_title(1, "📊 Dashboard")
tabs.set_title(2, "💬 Chatbot")

show_login_screen()
display(tabs)